# 4.7 训练可观测性 (Training Observability)

> 🕐 预估学习时间：40分钟

大规模训练的故障往往先表现为激活爆炸、梯度消失、Loss Spike 或某层“坏死”。训练可观测性把内部统计变成可告警信号，支撑回滚与根因定位。

本节涵盖：
- 激活 / 梯度 / 权重健康看板
- Loss 归因到样本与层
- Spike 检测与自动处置
- 与检查点 / 弹性训练联动


## 1. 激活与梯度健康指标

对每层记录：激活均值/方差/最大绝对值、梯度范数、参数更新比例。异常模式常早于 loss 发散出现。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict

torch.manual_seed(0)


class TinyStack(nn.Module):
    def __init__(self, d=64, n=4, vocab=50):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.layers = nn.ModuleList([nn.Linear(d, d) for _ in range(n)])
        self.head = nn.Linear(d, vocab)

    def forward(self, x, collect=False):
        stats = {}
        h = self.embed(x)
        for i, layer in enumerate(self.layers):
            h = F.gelu(layer(h))
            if collect:
                stats[f'L{i}.act'] = {
                    'mean': h.mean().item(),
                    'std': h.std().item(),
                    'absmax': h.abs().max().item(),
                    'nan': torch.isnan(h).any().item(),
                }
        return self.head(h.mean(1)), stats


def grad_norms(model):
    out = {}
    for n, p in model.named_parameters():
        if p.grad is not None:
            out[n] = p.grad.detach().norm().item()
    return out


model = TinyStack()
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
print('=== Activation / Gradient Health ===')
for step in range(8):
    x = torch.randint(0, 50, (16, 12))
    y = torch.randint(0, 50, (16,))
    # inject a bad batch mid-way
    if step == 4:
        x = x * 0 + 49  # pathological
    logits, stats = model(x, collect=True)
    loss = F.cross_entropy(logits, y)
    # amplify loss spike
    if step == 4:
        loss = loss * 20
    opt.zero_grad()
    loss.backward()
    g = grad_norms(model)
    opt.step()
    absmax = max(v['absmax'] for v in stats.values())
    gmax = max(g.values()) if g else 0
    flag = []
    if absmax > 50:
        flag.append('ACT_EXPLODE')
    if gmax > 100:
        flag.append('GRAD_EXPLODE')
    if any(v['nan'] for v in stats.values()):
        flag.append('NAN')
    print(f'step={step} loss={loss.item():.3f} act_absmax={absmax:.2f} grad_max={gmax:.2f} {flag}')
print(f'\nKey: Per-layer absmax/grad norms catch pathologies before total collapse.')


## 2. Loss 归因：是哪类样本 / 哪一层？

- **样本归因**：高 loss 样本的领域/长度/语言分布  
- **层归因**：阻断某层残差写入后 loss 变化（类似 patching）


In [ ]:
def sample_loss_attribution(model, x, y):
    model.eval()
    with torch.no_grad():
        logits, _ = model(x, collect=False)
        per = F.cross_entropy(logits, y, reduction='none')
    model.train()
    order = per.argsort(descending=True)
    return per, order


def layer_ablation_delta(model, x, y):
    '''Zero-out each layer output once and measure CE delta.'''
    base_logits, _ = model(x, collect=False)
    base = F.cross_entropy(base_logits, y).item()
    deltas = []
    hooks = []

    def make_hook():
        def hook(_m, _inp, out):
            return torch.zeros_like(out)
        return hook

    for i, layer in enumerate(model.layers):
        h = layer.register_forward_hook(make_hook())
        logits, _ = model(x, collect=False)
        loss = F.cross_entropy(logits, y).item()
        deltas.append((i, loss - base))
        h.remove()
    return base, deltas


x = torch.randint(0, 50, (32, 10))
y = torch.randint(0, 50, (32,))
per, order = sample_loss_attribution(model, x, y)
base, deltas = layer_ablation_delta(model, x[:8], y[:8])
print('=== Loss Attribution ===')
print(f'top-5 sample losses: {per[order[:5]].tolist()}')
print(f'layer ablation deltas: {deltas}')
print(f'\nKey: High-loss samples + sensitive layers localize data bugs vs architecture bugs.')


## 3. Spike 检测与自动处置

规则示例：滚动窗口 z-score / 相对跳变；触发后：跳过 batch、降 LR、回滚检查点、告警。


In [ ]:
class SpikeGuard:
    def __init__(self, window=20, z_thr=4.0, ratio_thr=2.5):
        self.window = window
        self.z_thr = z_thr
        self.ratio_thr = ratio_thr
        self.hist = []
        self.actions = []

    def update(self, loss, step):
        import statistics
        self.hist.append(loss)
        if len(self.hist) < 5:
            return 'ok'
        w = self.hist[-self.window:]
        mu = statistics.fmean(w[:-1])
        sd = statistics.pstdev(w[:-1]) or 1e-6
        z = (loss - mu) / sd
        ratio = loss / max(mu, 1e-6)
        if z > self.z_thr or ratio > self.ratio_thr:
            action = 'skip_batch+rollback_candidate'
            self.actions.append((step, loss, z, ratio, action))
            # do not keep spike in baseline
            self.hist.pop()
            return action
        return 'ok'


guard = SpikeGuard()
losses = [2.1, 2.0, 1.95, 1.9, 1.88, 1.85, 1.84, 1.83, 8.5, 1.82, 1.81]
print('=== Spike Guard ===')
for i, loss in enumerate(losses):
    print(f'step={i} loss={loss} -> {guard.update(loss, i)}')
print('recorded:', guard.actions)
print(f'\nKey: Treat spikes as first-class events with skip/rollback, not just dashboard red lines.')


## 4. 看板字段清单（产业）

| 类别 | 指标 | 告警 |
|------|------|------|
| 标量 | loss/lr/grad_norm/tokens/s | z-score / SLA |
| 张量 | 层激活 absmax、注意力熵 | 阈值 |
| 数据 | 坏 batch 指纹、领域占比 | 突增 |
| 系统 | MFU、通信等待、OOMS | 趋势 |

与 W&B/MLflow/Prometheus 对接时，保留 **可复现 run_id + checkpoint 指针**。


In [ ]:
dashboard = {
    'loss': 1.82,
    'grad_norm': 12.4,
    'act_absmax_p99': 18.2,
    'mfu': 0.41,
    'tokens_per_sec': 1.2e5,
    'spike_count_1h': 1,
}
print('=== Dashboard Snapshot ===')
for k, v in dashboard.items():
    print(f'{k}: {v}')
print(f'\nKey: Observability is useful only when metrics map to concrete remediation playbooks.')


## 课后思考题

1. 激活 absmax 升高但 loss 仍降，何时需要干预？
2. 样本归因如何避免把“本来就难的题”误判为脏数据？
3. 多机训练下如何聚合层统计而不拖垮吞吐？
4. Spike 回滚与跳过 batch 的取舍如何影响最终模型质量？

---
> 本节涵盖了4.7 训练可观测性的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
